# Group 12 — PLSA Information Retrieval Project (All 20 Categories)
This notebook trains PLSA on **all 20 categories** of 20 Newsgroups. It implements PLSA with NumPy/EM, inspects topics, tests controlled mixtures from two selected categories, performs query folding-in, and compares retrieval against TF-IDF/VSM and LSI.

**Main experiment:** all 20 categories. **Boundary experiment:** a controlled mixture of two source categories (baseball and politics), as specified in the proposal. Category labels are not used to train PLSA; they are used for evaluation and selecting synthetic source documents.


## 0. Install dependencies

Run once in a notebook environment if required. In Colab, use the first line in a code cell.

In [ ]:
# %pip -q install numpy scipy scikit-learn pandas matplotlib

## 1. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.sparse import csr_matrix
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import Normalizer
from sklearn.metrics.pairwise import cosine_similarity

SEED = 42
rng = np.random.default_rng(SEED)
np.random.seed(SEED)


## 2. Load the selected dataset

We load only the two categories selected for the topic-boundary experiment. The category labels are used for evaluation and for choosing source documents for synthetic mixtures; PLSA itself is unsupervised and does not receive these labels during training.

In [ ]:
dataset = fetch_20newsgroups(
    subset="all",
    remove=("headers", "footers", "quotes"),
    shuffle=True,
    random_state=SEED
)
documents = np.array(dataset.data, dtype=object)
labels = np.asarray(dataset.target)
target_names = list(dataset.target_names)

print("Number of categories:", len(target_names))
print("Total documents:", len(documents))
for i, name in enumerate(target_names):
    print(f"{i:2d} | {name:25s} | {(labels == i).sum():5d} documents")

### Main corpus versus boundary experiment

The model and retrieval evaluation use the **full 20-category corpus**. The controlled boundary experiment still uses two chosen source categories to create interpretable 90:10 through 10:90 word mixtures. This does not mean the PLSA model was trained on only those two categories.

## 3. Train/test split and preprocessing

Split the raw documents first. Fit the vocabulary only on the training split to avoid vocabulary leakage from the test documents.

In [ ]:
import re
import spacy
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

docs_train, docs_test, y_train, y_test = train_test_split(
    documents,
    labels,
    test_size=0.25,
    random_state=SEED,
    stratify=labels
)
nlp = spacy.load(
    "en_core_web_sm",
    disable=["ner", "parser"]
)
def preprocess_texts(texts):
    """
    Lowercase, tokenize, remove stop words and punctuation,
    and lemmatize the remaining tokens.

    Returns a list of preprocessed document strings.
    """
    processed_docs = []

    for doc in nlp.pipe(texts, batch_size=256):
        tokens = []

        for token in doc:

            if (
                token.is_space
                or token.is_punct
                or token.like_num
                or not token.is_alpha
            ):
                continue


            if token.is_stop:
                continue


            lemma = token.lemma_.lower().strip()


            if len(lemma) < 2:
                continue

            tokens.append(lemma)

        processed_docs.append(" ".join(tokens))

    return processed_docs
docs_train_processed = preprocess_texts(docs_train)
docs_test_processed = preprocess_texts(docs_test)
count_vectorizer = CountVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    max_features=3000,
    min_df=3,
    max_df=0.90,
    dtype=np.float64
)
X_train = count_vectorizer.fit_transform(
    docs_train_processed
)
X_test = count_vectorizer.transform(
    docs_test_processed
)
vocabulary = count_vectorizer.get_feature_names_out()
print("X_train:", X_train.shape, "(documents, vocabulary)")
print("X_test: ", X_test.shape)
print("Example vocabulary:", vocabulary[:20])

print(
    "Empty training documents after preprocessing:",
    sum(not doc.strip() for doc in docs_train_processed)
)

print(
    "Empty test documents after preprocessing:",
    sum(not doc.strip() for doc in docs_test_processed)
)


## 4. PLSA theory

For document \(d\), word \(w\), and latent topic \(z\):

\[
P(w\mid d)=\sum_{z=1}^{K}P(w\mid z)P(z\mid d)
\]

- \(\theta_{d,z}=P(z\mid d)\): document-topic distribution.
- \(\phi_{z,w}=P(w\mid z)\): topic-word distribution.
- Each row of \(\theta\) and \(\phi\) sums to one.

For every observed document-word pair, the E-step computes the posterior responsibility:

\[
P(z\mid d,w)=\frac{P(z\mid d)P(w\mid z)}
{\sum_{z'}P(z'\mid d)P(w\mid z')}
\]

The M-step uses the expected word counts to update both distributions. The process repeats until the log-likelihood stops improving.

In [ ]:
class PLSA:
    """PLSA trained directly with NumPy EM and sparse input counts."""

    def __init__(self, n_topics=5, max_iter=100, tol=1e-4,
                 random_state=42, beta=1.0):
        if n_topics < 1:
            raise ValueError("n_topics must be positive")
        if not (0 < beta):
            raise ValueError("beta must be positive")
        self.n_topics = n_topics
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.beta = beta
        self.log_likelihoods = []

    @staticmethod
    def _normalize(a, axis):
        denom = np.maximum(a.sum(axis=axis, keepdims=True), 1e-15)
        return a / denom

    @staticmethod
    def _observations(X):
        X = csr_matrix(X, dtype=np.float64)
        X.eliminate_zeros()
        X.sum_duplicates()
        docs, words, counts = [], [], []
        for d in range(X.shape[0]):
            start, end = X.indptr[d], X.indptr[d + 1]
            if end > start:
                docs.append(np.full(end-start, d, dtype=np.int64))
                words.append(X.indices[start:end].astype(np.int64))
                counts.append(X.data[start:end].astype(np.float64))
        if not docs:
            raise ValueError("Input has no observed word counts.")
        return X, np.concatenate(docs), np.concatenate(words), np.concatenate(counts)

    def fit(self, X):
        X, doc_ids, word_ids, counts = self._observations(X)
        D, V = X.shape
        K = self.n_topics
        rng = np.random.default_rng(self.random_state)

        theta = self._normalize(rng.random((D, K)) + 1e-3, axis=1)
        phi = self._normalize(rng.random((K, V)) + 1e-3, axis=1)
        self.log_likelihoods = []

        for iteration in range(self.max_iter):

            unnorm = theta[doc_ids].T * phi[:, word_ids]
            if self.beta != 1.0:
                unnorm = np.maximum(unnorm, 1e-15) ** self.beta
            posterior = unnorm / np.maximum(
                unnorm.sum(axis=0, keepdims=True), 1e-15
            )


            weighted = posterior * counts[None, :]
            topic_word = np.zeros((K, V), dtype=np.float64)
            doc_topic = np.zeros((D, K), dtype=np.float64)

            for k in range(K):
                np.add.at(topic_word[k], word_ids, weighted[k])
                np.add.at(doc_topic[:, k], doc_ids, weighted[k])

            phi = self._normalize(topic_word + 1e-12, axis=1)
            theta = self._normalize(doc_topic + 1e-12, axis=1)


            p_observed = np.sum(theta[doc_ids] * phi[:, word_ids].T, axis=1)
            ll = float(np.sum(counts * np.log(np.maximum(p_observed, 1e-15))))
            self.log_likelihoods.append(ll)

            if iteration > 0:
                improvement = ll - previous_ll
                if self.beta == 1.0 and improvement < -1e-7:
                    print("Warning: likelihood decreased slightly; check numerical behavior.")
                if abs(improvement) <= self.tol * (1 + abs(previous_ll)):
                    break
            previous_ll = ll

        self.theta_ = theta
        self.phi_ = phi
        self.n_features_in_ = V
        self.n_iter_ = len(self.log_likelihoods)
        return self

    def transform(self):
        """Document-topic probabilities for the documents used in fit()."""
        return self.theta_

    def fold_in(self, X_new, max_iter=100):
        """Infer P(z|d_new) with learned P(w|z) fixed."""
        X_new = csr_matrix(X_new, dtype=np.float64)
        if X_new.shape[1] != self.n_features_in_:
            raise ValueError("New data must use the same fitted vocabulary.")
        D, V = X_new.shape
        K = self.n_topics
        rng = np.random.default_rng(self.random_state + 1000)
        theta = self._normalize(rng.random((D, K)) + 1e-3, axis=1)

        for _ in range(max_iter):
            doc_topic = np.zeros((D, K), dtype=np.float64)
            for d in range(D):
                start, end = X_new.indptr[d], X_new.indptr[d + 1]
                words = X_new.indices[start:end]
                counts = X_new.data[start:end]
                if len(words) == 0:
                    continue
                unnorm = theta[d, :, None] * self.phi_[:, words]
                if self.beta != 1.0:
                    unnorm = np.maximum(unnorm, 1e-15) ** self.beta
                posterior = unnorm / np.maximum(
                    unnorm.sum(axis=0, keepdims=True), 1e-15
                )
                doc_topic[d] = (posterior * counts[None, :]).sum(axis=1)
            theta = self._normalize(doc_topic + 1e-12, axis=1)
        return theta

    def top_words(self, vocabulary, n=10):
        result = {}
        for k, dist in enumerate(self.phi_):
            ids = np.argsort(dist)[::-1][:n]
            result[k] = [(vocabulary[i], float(dist[i])) for i in ids]
        return result

## 5. Multiple random restarts

EM can settle at different local optima. Fit several initializations and select the model with the largest training log-likelihood. This selection criterion is only about training fit; final retrieval quality must still be evaluated on held-out documents.

In [ ]:
def train_plsa_restarts(X, n_topics=5, n_restarts=5,
                        max_iter=100, tol=1e-4, base_seed=42):
    runs = []
    for r in range(n_restarts):
        model = PLSA(
            n_topics=n_topics, max_iter=max_iter, tol=tol,
            random_state=base_seed + r, beta=1.0
        ).fit(X)
        final_ll = model.log_likelihoods[-1]
        runs.append((model, final_ll))
        print(f"Restart {r+1}: final training log-likelihood={final_ll:.3f}, "
              f"iterations={model.n_iter_}")
    best_model, best_ll = max(runs, key=lambda item: item[1])
    print("Selected best training likelihood:", round(best_ll, 3))
    return best_model, runs

K = 5
best_plsa, restart_runs = train_plsa_restarts(
    X_train, n_topics=K, n_restarts=5, max_iter=100
)
theta_train = best_plsa.transform()
theta_test = best_plsa.fold_in(X_test, max_iter=100)
print("theta_train:", theta_train.shape)
print("theta_test:", theta_test.shape)

## 6. Inspect topics and convergence

Topic IDs are arbitrary: topic 0 is not inherently sports or politics. Interpret each topic by its highest-probability words, and use category-level averages only as a diagnostic.

In [ ]:
for topic_id, words in best_plsa.top_words(vocabulary, n=12).items():
    print(f"\nTopic {topic_id}:")
    print(", ".join(f"{word} ({prob:.3f})" for word, prob in words))

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_plsa.log_likelihoods)+1),
         best_plsa.log_likelihoods, marker="o", markersize=3)
plt.xlabel("EM iteration")
plt.ylabel("Training log-likelihood")
plt.title("PLSA convergence")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Controlled topic-boundary experiment

Generate artificial documents by sampling word tokens from source documents in known categories. We then infer topic mixtures with folding-in and compare the known source proportions with the estimated probabilities.

**Interpretation caution:** a source-word ratio is not guaranteed to equal a latent-topic probability. Topics can split a category or overlap categories. We therefore identify sports-associated and politics-associated topics from training-set category averages, and interpret the result as an experiment rather than ground truth equivalence.

In [ ]:
sports_class = target_names.index("rec.sport.baseball")
politics_class = target_names.index("talk.politics.misc")

class_topic_means = np.vstack([
    theta_train[y_train == c].mean(axis=0)
    for c in range(len(target_names))
])
sports_topic = int(class_topic_means[sports_class].argmax())
politics_topic = int(class_topic_means[politics_class].argmax())

print("Sports-associated topic:", sports_topic)
print("Politics-associated topic:", politics_topic)
if sports_topic == politics_topic:
    print("Warning: both categories map to the same dominant topic; interpret estimates cautiously.")

def sample_token_ids(row):
    row = csr_matrix(row)
    return np.repeat(row.indices, row.data.astype(int))

def make_mixed_document(X_source, y_source, class_a, class_b,
                        ratio_a, rng, n_tokens=300):

    a_candidates = np.where((y_source == class_a) & (np.asarray(X_source.sum(axis=1)).ravel() > 0))[0]
    b_candidates = np.where((y_source == class_b) & (np.asarray(X_source.sum(axis=1)).ravel() > 0))[0]

    da = X_source[a_candidates[rng.integers(len(a_candidates))]]
    db = X_source[b_candidates[rng.integers(len(b_candidates))]]
    tokens_a = sample_token_ids(da)
    tokens_b = sample_token_ids(db)
    if len(tokens_a) == 0 or len(tokens_b) == 0:
        raise ValueError("A sampled source document had no in-vocabulary tokens.")
    n_a = int(round(n_tokens * ratio_a))
    n_b = n_tokens - n_a
    tokens = np.concatenate([
        rng.choice(tokens_a, size=n_a, replace=True),
        rng.choice(tokens_b, size=n_b, replace=True)
    ])
    counts = np.bincount(tokens, minlength=X_source.shape[1]).astype(float)
    return csr_matrix(counts.reshape(1, -1))

boundary_rng = np.random.default_rng(SEED + 77)
ratios = [0.9, 0.7, 0.5, 0.3, 0.1]
boundary_rows = []

for ratio in ratios:
    for replicate in range(20):
        mixed_X = make_mixed_document(
            X_train, y_train, sports_class, politics_class,
            ratio, boundary_rng, n_tokens=300
        )
        estimate = best_plsa.fold_in(mixed_X, max_iter=100)[0]
        boundary_rows.append({
            "source_category_a": "rec.sport.baseball",
            "source_category_b": "talk.politics.misc",
            "actual_sports_word_ratio": ratio,
            "actual_politics_word_ratio": 1-ratio,
            "estimated_sports_topic": estimate[sports_topic],
            "estimated_politics_topic": estimate[politics_topic],
            "replicate": replicate
        })

boundary_df = pd.DataFrame(boundary_rows)
boundary_summary = boundary_df.groupby("actual_sports_word_ratio").agg(
    estimated_sports_mean=("estimated_sports_topic", "mean"),
    estimated_sports_std=("estimated_sports_topic", "std"),
    estimated_politics_mean=("estimated_politics_topic", "mean"),
    estimated_politics_std=("estimated_politics_topic", "std")
).reset_index()
display(boundary_summary)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot([0, 1], [0, 1], "k--", label="Ideal reference")
plt.errorbar(
    boundary_summary["actual_sports_word_ratio"],
    boundary_summary["estimated_sports_mean"],
    yerr=boundary_summary["estimated_sports_std"],
    marker="o", capsize=3, label="Estimated sports-topic probability"
)
plt.xlabel("Actual sports word-token proportion")
plt.ylabel("Estimated sports-associated topic probability")
plt.title("PLSA topic-boundary experiment")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. TF-IDF/VSM and LSI baselines

All baselines are fitted on training documents only. LSI uses truncated SVD on TF-IDF representations; it is a geometric low-rank representation, unlike PLSA's normalized probability distributions.

In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=False,
    tokenizer=str.split,
    token_pattern=None,
    max_features=3000,
    min_df=3,
    max_df=0.90
)

X_train_tfidf = tfidf_vectorizer.fit_transform(docs_train_processed)
X_test_tfidf = tfidf_vectorizer.transform(docs_test_processed)

n_components = min(50, X_train_tfidf.shape[0]-1, X_train_tfidf.shape[1]-1)
lsi_model = make_pipeline(
    TruncatedSVD(n_components=n_components, random_state=SEED),
    Normalizer(copy=False)
)
lsi_train = lsi_model.fit_transform(X_train_tfidf)
lsi_test = lsi_model.transform(X_test_tfidf)

print("TF-IDF train/test:", X_train_tfidf.shape, X_test_tfidf.shape)
print("LSI train/test:", lsi_train.shape, lsi_test.shape)

## 9. Retrieval metrics and fair comparison

For each query, relevance is defined using the known test-category labels. The same test collection and relevance class are used for PLSA, LSI and TF-IDF. This is a small demonstration with category queries; a stronger report should evaluate a broader, predeclared query set and report macro averages.

**NDCG@10 replaces Recall@10 and F1@10** in this evaluation. Each document has binary relevance (1 if its label matches the query category; 0 otherwise). NDCG rewards relevant documents near the top and normalizes against an ideal ranking. With binary relevance, it does not distinguish degrees of relevance within the same category.


Evaluation reports **Precision@10, Recall@10, NDCG@10, and MAP@10**. MAP@10 is the average of AP@10 over all category queries.

In [ ]:
category_queries = {
    "alt.atheism": "atheism atheist religion belief",
    "comp.graphics": "computer graphics rendering image pixels",
    "comp.os.ms-windows.misc": "windows operating system microsoft files",
    "comp.sys.ibm.pc.hardware": "IBM PC hardware disk memory computer",
    "comp.sys.mac.hardware": "Apple Macintosh Mac hardware computer",
    "comp.windows.x": "X Window System display server windows",
    "misc.forsale": "for sale price shipping offer",
    "rec.autos": "car automobile engine dealer",
    "rec.motorcycles": "motorcycle bike riding helmet",
    "rec.sport.baseball": "baseball players team season game",
    "rec.sport.hockey": "hockey team NHL puck season",
    "sci.crypt": "cryptography encryption key security",
    "sci.electronics": "electronics circuit voltage resistor",
    "sci.med": "medical doctor disease treatment health",
    "sci.space": "space NASA spacecraft orbit planet",
    "soc.religion.christian": "Christian church Bible Jesus faith",
    "talk.politics.guns": "gun control firearms second amendment",
    "talk.politics.mideast": "Middle East Israel Arab Palestinian",
    "talk.politics.misc": "government election political policy",
    "talk.religion.misc": "religion religious belief faith"
}

missing = sorted(set(target_names) - set(category_queries))
extra = sorted(set(category_queries) - set(target_names))
print("Categories without a query:", missing)
print("Query names not found in dataset:", extra)

def evaluate_query(query_text, target_label, k=10):


    processed_query = preprocess_texts([query_text])[0]
    if not processed_query.strip():
        print(f"Warning: query for {target_names[target_label]} became empty after preprocessing.")


    q_tfidf = tfidf_vectorizer.transform([processed_query])
    q_lsi = lsi_model.transform(q_tfidf)


    q_counts = count_vectorizer.transform([processed_query]).astype(np.float64)
    q_plsa = best_plsa.fold_in(q_counts, max_iter=100)


    sim_tfidf = cosine_similarity(q_tfidf, X_test_tfidf).ravel()
    sim_lsi = cosine_similarity(q_lsi, lsi_test).ravel()
    sim_plsa = cosine_similarity(q_plsa, theta_test).ravel()

    methods = {
        "TF-IDF/VSM": sim_tfidf,
        "LSI": sim_lsi,
        "PLSA": sim_plsa
    }

    rows = []
    total_relevant = int((y_test == target_label).sum())
    if total_relevant == 0:
        return rows

    for method_name, similarities in methods.items():

        top_indices = np.argsort(similarities)[::-1][:k]


        true_positives = int((y_test[top_indices] == target_label).sum())

        precision = true_positives / k
        # Binary relevance: a document is relevant if it belongs to the query's category.
        # DCG@k discounts lower-ranked relevant documents; IDCG@k is the ideal ranking.
        relevance = (y_test[top_indices] == target_label).astype(float)
        discounts = np.log2(np.arange(2, len(relevance) + 2))
        dcg = float(np.sum(relevance / discounts))
        ideal_count = min(k, total_relevant, len(relevance))
        idcg = float(np.sum(1.0 / np.log2(np.arange(2, ideal_count + 2))))
        ndcg = dcg / idcg if idcg > 0 else 0.0
        recall = true_positives / total_relevant
        precisions_at_hits = np.cumsum(relevance) / np.arange(1, len(relevance) + 1)
        ap_at_k = float(np.sum(precisions_at_hits * relevance) / min(total_relevant, k))
        rows.append({
            "query": query_text,
            "target_category": target_names[target_label],
            "method": method_name,
            "Precision@k": precision,
            "NDCG@k": ndcg,
            "Recall@k": recall,
            "AP@k": ap_at_k
        })
    return rows

eval_rows = []
for category_name in target_names:
    eval_rows.extend(
        evaluate_query(
            category_queries[category_name],
            target_names.index(category_name),
            k=10
        )
    )

evaluation_df = pd.DataFrame(eval_rows)
display(evaluation_df)
print("\nMean metrics across all 20 category queries:")
display(evaluation_df.groupby("method")[["Precision@k", "Recall@k", "NDCG@k", "AP@k"]].mean().rename(columns={"AP@k": "MAP@k"}))

In [ ]:
mean_metrics = evaluation_df.groupby("method")[["Precision@k", "Recall@k", "NDCG@k", "AP@k"]].mean().rename(columns={"AP@k": "MAP@k"})
mean_metrics.plot(kind="bar", figsize=(9, 5), ylim=(0, 1), rot=0)
plt.title("Held-out retrieval comparison")
plt.ylabel("Score")
plt.xlabel("Method")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Model-order study: how does K affect the model?

The proposal asks us to study how the number of latent topics affects convergence and topic separation. Compare a few values of K. For a fair held-out comparison, infer test-document topic distributions with `fold_in`; don't use training-document representations to score test documents.

In [ ]:
k_rows = []
for k_topics in [2, 3, 5, 8, 10]:
    model = PLSA(
        n_topics=k_topics, max_iter=80, tol=1e-4,
        random_state=SEED, beta=1.0
    ).fit(X_train)
    k_rows.append({
        "K": k_topics,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "test_documents": X_test.shape[0]
    })

k_df = pd.DataFrame(k_rows)
display(k_df)

## 11. Optional extension: tempered E-step (exploratory)

Hofmann's paper introduces Tempered EM as an entropy-regularized fitting procedure and recommends evaluating held-out performance while changing the inverse temperature. The `beta` option in the class is a simplified tempered E-step, **not a full reproduction of the paper's TEM schedule**. Treat this section as exploratory only, and don't claim that it fully implements TEM.

For a proper TEM experiment, start with standard EM at beta=1, lower beta in small steps, run tempered iterations, and monitor held-out predictive likelihood/perplexity; stop when held-out performance deteriorates. A rigorous implementation should define and document the exact schedule and stopping rule.

In [ ]:
tempered_rows = []
for beta in [1.0, 0.9, 0.8, 0.7]:
    model = PLSA(
        n_topics=K, max_iter=80, tol=1e-4,
        random_state=SEED, beta=beta
    ).fit(X_train)
    tempered_rows.append({
        "beta": beta,
        "iterations": model.n_iter_,
        "train_log_likelihood": model.log_likelihoods[-1],
        "warning": "Simplified tempered E-step; not full TEM"
    })

display(pd.DataFrame(tempered_rows))

## 12. Limitations to state in the report

1. PLSA is trained on all 20 categories, but learned topics do not necessarily correspond one-to-one to the 20 human labels.
2. Synthetic source-word proportions in the baseball/politics boundary experiment are not identical to latent-topic probabilities.
3. Selecting the best training likelihood does not guarantee the best retrieval performance.
4. Category labels are only a proxy for relevance; manually judged query-document relevance would be stronger.
5. The 20 representative queries are manually authored, and query wording can affect ranking. Include the query set in the report.
6. Hofmann's paper used MED, CRAN, CACM and CISI collections; this project uses 20 Newsgroups and does not reproduce the paper's exact numerical results.
7. The optional beta experiment is a simplified tempered E-step unless the full TEM schedule and held-out stopping criterion are implemented.
8. A capped vocabulary and sparse count matrix are used to keep computation manageable.